# 06 — Streaming

## Goal

Understand streaming at three independent layers:

1. Model streaming — tokens generated by the LLM
2. LangGraph / Deep Agent streaming — progress through agent steps
3. Responses protocol streaming — SSE events delivered to the client

We will first inspect Deep Agent streaming locally.

We will then map useful agent output into the Foundry Responses streaming
protocol.

We will not add checkpoint persistence or background execution yet.

```
Client
  │
  │ Responses SSE events
  ▼
Hosted Agent handler
  │
  │ LangGraph stream
  ▼
Deep Agent
  │
  │ model token stream
  ▼
LLM

In [2]:
from deep_agents_foundry import build_research_agent

agent = build_research_agent()

stream = agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Explain Microsoft Foundry Hosted Agents in 3 bullets.",
            }
        ]
    },
    stream_mode="updates",
)

print(type(stream))

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()


<class 'generator'>


In [3]:
for chunk in stream:
    print("\n--- CHUNK ---")
    print(chunk)


--- CHUNK ---
{'PatchToolCallsMiddleware.before_agent': None}

--- CHUNK ---
{'model': {'messages': [AIMessage(content=[{'id': 'ws_0d6da3c937452ec2006aa1a2b0f5b4819096ab1c3e44c60468', 'action': {'type': 'search', 'queries': ['Microsoft Foundry Hosted Agents', '"Foundry" "Hosted Agents" Microsoft', 'Azure AI Foundry hosted agents', 'Microsoft Foundry Agents hosted'], 'query': 'Microsoft Foundry Hosted Agents'}, 'status': 'completed', 'type': 'web_search_call', 'response_id': 'resp_0d6da3c937452ec2006aa1a2b015008190aa4188c046a7957f'}, {'type': 'text', 'text': '- **What it is:** *Microsoft Foundry Hosted Agents* is a managed hosting option in **Foundry Agent Service** that lets you run your agent (often as a **containerized app**) on Microsoft-managed infrastructure, while your agent uses models from the **Foundry model catalog** for reasoning. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents))\n\n- **What Microsoft manages for you:** T

## `updates` = agent progress

`updates` does not primarily mean token-by-token text.

It emits changes after graph/agent steps.

For an agent with tools, a run may look like:

```
model
  ↓
tool request

tool
  ↓
tool result

model
  ↓
final answer

In [6]:
for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Research the current Microsoft Foundry Hosted Agent model "
                    "and explain the key operational responsibilities Foundry manages."
                ),
            }
        ]
    },
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        print(f"\nNODE: {node_name}")
        print(update)


NODE: PatchToolCallsMiddleware.before_agent
None

NODE: model
{'messages': [AIMessage(content=[{'id': 'ws_0bd5262d8dcaba3d006aa1a48ce208819484eaaf5354b3f34d', 'action': {'type': 'search', 'queries': ['Microsoft Foundry Hosted Agent model operational responsibilities Foundry manages', '"Foundry" "Hosted Agent" Microsoft responsibilities managed by Foundry', 'Microsoft Foundry hosted agent documentation', 'Azure AI Foundry hosted agent model what Foundry manages'], 'query': 'Microsoft Foundry Hosted Agent model operational responsibilities Foundry manages'}, 'status': 'completed', 'type': 'web_search_call', 'response_id': 'resp_0bd5262d8dcaba3d006aa1a48b96648194b02e15711c6abe9a'}, {'id': 'ws_0bd5262d8dcaba3d006aa1a48fbc888194a25610b655db3119', 'action': {'type': 'open_page', 'url': 'https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents'}, 'status': 'completed', 'type': 'web_search_call', 'response_id': 'resp_0bd5262d8dcaba3d006aa1a48b96648194b02e15711c6abe9a'}, {

## Streaming vs tracing

Streaming answers:

> What can the caller observe while the task is executing?

Tracing answers:

> What happened operationally during the run?

Streaming is part of the interaction contract.

Tracing is observability.

In [22]:
# for chunk in agent.stream(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": "Explain Hosted Agent identity briefly.",
#             }
#         ]
#     },
#     stream_mode="messages",
# ):
#     print(chunk)
#     print()

In [23]:
# # Current LangGraph docs expose a cleaner stable envelope with:

# for part in agent.stream(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": "Explain Hosted Agent identity briefly.",
#             }
#         ]
#     },
#     stream_mode="messages",
#     version="v2",
# ):
#     print(part)
#     print()

In [26]:
def extract_text_from_chunk(message_chunk) -> str:
    blocks = getattr(message_chunk, "content_blocks", None) or []

    text_parts = []

    for block in blocks:
        if block.get("type") == "text" and block.get("text"):
            text_parts.append(block["text"])

    return "".join(text_parts)

In [ ]:
for part in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Explain Hosted Agent identity in one paragraph.",
            }
        ]
    },
    stream_mode="messages",
    version="v2",
):
    if part["type"] != "messages":
        continue

    message_chunk, metadata = part["data"]

    text = extract_text_from_chunk(message_chunk)

    if text:
        print(text, end="", flush=True)    
        
        

Hosted Agent identity is the **built-in identity and credential context** a managed (“hosted”) AI agent uses to securely act on your behalf when it calls tools, APIs, or other services—without you having to embed long‑lived secrets in prompts or code. Instead of the agent “being” an end user, the platform typically gives the agent its own **service identity** (and sometimes the ability to **impersonate** a user with explicit delegation) that is used to request short‑lived tokens, authenticate outbound requests, and enforce **least‑privilege** access via policies (scopes/roles), auditing, and key rotation. In practice, this identity layer answers: *who is the agent*, *what is it allowed to do*, *how do downstream systems verify that*, and *how is usage logged and governed*—so the agent can reliably access permitted resources while keeping authentication centralized, revocable, and observable.

In [ ]:
# query = """
# What are the latest capabilities of Microsoft Foundry Hosted Agents?
# Use current Microsoft documentation.
# """

query = "Explain Hosted Agent identity in one paragraph."

# for part in agent.stream(
#     {"messages": [{"role": "user", "content": query}]},
#     stream_mode="messages",
#     version="v2",
# ):
#     if part["type"] != "messages":
#         continue

#     message_chunk, metadata = part["data"]

#     print("\nCHUNK TYPE:", type(message_chunk).__name__)
#     print("CONTENT BLOCKS:")

#     for block in getattr(message_chunk, "content_blocks", None) or []:
#         print(block)


CHUNK TYPE: AIMessageChunk
CONTENT BLOCKS:

CHUNK TYPE: AIMessageChunk
CONTENT BLOCKS:

CHUNK TYPE: AIMessageChunk
CONTENT BLOCKS:
{'type': 'text', 'text': 'Hosted', 'index': 'lc_txt_0'}

CHUNK TYPE: AIMessageChunk
CONTENT BLOCKS:
{'type': 'text', 'text': ' Agent', 'index': 'lc_txt_0'}

CHUNK TYPE: AIMessageChunk
CONTENT BLOCKS:
{'type': 'text', 'text': ' identity', 'index': 'lc_txt_0'}

CHUNK TYPE: AIMessageChunk
CONTENT BLOCKS:
{'type': 'text', 'text': ' is', 'index': 'lc_txt_0'}

CHUNK TYPE: AIMessageChunk
CONTENT BLOCKS:
{'type': 'text', 'text': ' the', 'index': 'lc_txt_0'}

CHUNK TYPE: AIMessageChunk
CONTENT BLOCKS:
{'type': 'text', 'text': ' built', 'index': 'lc_txt_0'}

CHUNK TYPE: AIMessageChunk
CONTENT BLOCKS:
{'type': 'text', 'text': '-in', 'index': 'lc_txt_0'}

CHUNK TYPE: AIMessageChunk
CONTENT BLOCKS:
{'type': 'text', 'text': ' identity', 'index': 'lc_txt_0'}

CHUNK TYPE: AIMessageChunk
CONTENT BLOCKS:
{'type': 'text', 'text': ' and', 'index': 'lc_txt_0'}

CHUNK TYPE: AIM

In [37]:
for part in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode=["updates", "messages"],
    version="v2",
):
    if part["type"] == "messages":
        message_chunk, metadata = part["data"]

        text = extract_text_from_chunk(message_chunk)

        if text:
            print(text, end="", flush=True)

    elif part["type"] == "updates":
        print("\n\n[AGENT UPDATE]")
        print(part["data"])



[AGENT UPDATE]
{'PatchToolCallsMiddleware.before_agent': None}
Hosted Agent identity is the mechanism a “hosted” (provider-managed) AI agent uses to authenticate itself and be authorized when it calls tools, APIs, or enterprise resources on behalf of an application or organization—analogous to a service account for agents. Instead of relying on an end-user’s personal credentials, the hosted agent runs under its own managed identity (often with administratively defined roles/scopes), so downstream systems can verify “this call came from *that* agent,” enforce least-privilege access, rotate credentials/keys safely, and record auditable logs of what the agent accessed and did. This separation helps organizations control permissions, support compliance (traceability and policy enforcement), and reduce risk when agents operate autonomously across multiple systems.

[AGENT UPDATE]
{'model': {'messages': [AIMessage(content=[{'type': 'text', 'text': 'Hosted Agent identity is the mechanism a 

In [38]:
def inspect_stream(part):
    stream_type = part["type"]

    if stream_type == "updates":
        print("\n\n[AGENT UPDATE]")

        update = part["data"]

        for node_name, node_update in update.items():
            print(f"Node: {node_name}")
            print(node_update)

    elif stream_type == "messages":
        message_chunk, metadata = part["data"]

        text = extract_text_from_chunk(message_chunk)

        if text:
            print(text, end="", flush=True)

In [39]:
for part in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode=["updates", "messages"],
    version="v2",
):
    inspect_stream(part)



[AGENT UPDATE]
Node: PatchToolCallsMiddleware.before_agent
None
Hosted Agent identity is the **managed, platform-provided identity** assigned to an AI agent that runs on a hosting service (rather than on your own servers) so it can securely interact with external systems. Instead of baking long-lived API keys into prompts or code, the host typically provisions an identity (often backed by a cloud identity such as a service account / managed identity) and uses it to **authenticate and authorize** the agent when it calls tools, APIs, databases, or other resources; administrators can then enforce least-privilege permissions, rotate credentials automatically, scope access per agent/environment, and audit what the agent did. In practice, “hosted agent identity” is about giving an agent a controlled “who it is” for access control—separate from the agent’s conversational persona—so that actions the agent takes can be securely permitted, traced, and governed.

[AGENT UPDATE]
Node: model
{'me

## Streaming != parallel execution

Streaming changes when results become observable.

It does not automatically make agent execution faster or more parallel.

Example:

search 1 takes 10 sec
search 2 takes 10 sec
model synthesis takes 20 sec

Streaming can surface progress during those 40 seconds.

It does not necessarily reduce the 40 seconds.